In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [5]:
TRAIN_DIR = Path("../datasets/raw/astro_dataset_maxia/training")
VALID_DIR = Path("../datasets/raw/astro_dataset_maxia/validation")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=transform
)

valid_dataset = datasets.ImageFolder(
    VALID_DIR,
    transform=transform
)

print("Training Images :", len(train_dataset))
print("Validation Images:", len(valid_dataset))
print("Classes:", train_dataset.classes)

Training Images : 2416
Validation Images: 658
Classes: ['asteroid', 'black_hole', 'earth', 'galaxy', 'jupiter', 'mars', 'mercury', 'neptune', 'pluto', 'saturn', 'uranus', 'venus']


In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

In [9]:
weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Diksha Tahilyani/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:11<00:00, 3.96MB/s]

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [11]:
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(train_dataset.classes)
)

model = model.to(device)

print(model.fc)

Linear(in_features=512, out_features=12, bias=True)


In [12]:
for param in model.parameters():
    param.requires_grad = False

# unfreeze only the final layer
for param in model.fc.parameters():
    param.requires_grad = True

In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print(criterion)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [14]:
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}")

Epoch [1/5] Loss: 1.0401
Epoch [2/5] Loss: 0.2907
Epoch [3/5] Loss: 0.1852
Epoch [4/5] Loss: 0.1481
Epoch [5/5] Loss: 0.1111


In [15]:
model.eval()

running_val_loss = 0.0

correct = 0
total = 0

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        running_val_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

val_loss = running_val_loss / len(valid_loader)

accuracy = 100 * correct / total

print("=" * 50)
print("ResNet18 Validation Results")
print("=" * 50)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {accuracy:.2f}%")
print("=" * 50)

ResNet18 Validation Results
Validation Loss     : 0.0863
Validation Accuracy : 98.02%


In [16]:
MODEL_PATH = Path("../models/resnet18_transfer.pth")

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print(f"Model saved to: {MODEL_PATH}")

Model saved to: ..\models\resnet18_transfer.pth


In [17]:
print("-" * 60)
print(f"{'Model':<20}{'Validation Accuracy'}")
print("-" * 60)
print(f"{'Simple CNN':<20}{92.10:.2f}%")
print(f"{'ResNet18':<20}{accuracy:.2f}%")
print("-" * 60)

------------------------------------------------------------
Model               Validation Accuracy
------------------------------------------------------------
Simple CNN          92.10%
ResNet18            98.02%
------------------------------------------------------------


# Conclusion

## Objective
To improve image classification performance using Transfer Learning with a pretrained ResNet18 model.

## What We Did
- Loaded a pretrained ResNet18 model.
- Replaced the final classification layer.
- Froze pretrained feature extraction layers.
- Fine-tuned the classifier on the astronomy dataset.
- Compared performance with the custom CNN.

## Results
| Model | Validation Accuracy |
|--------|--------------------:|
| Simple CNN | **92.10%** |
| ResNet18 | **98.02%** |

## Key Learnings
- Transfer Learning.
- Pretrained neural networks.
- Feature extraction.
- Layer freezing.
- Fine-tuning.
- Performance comparison.

## Conclusion
Transfer Learning significantly improved classification performance while reducing the need to learn visual features from scratch. The pretrained ResNet18 achieved 98.02% validation accuracy, outperforming the custom CNN.